# Single-stock risk report
Prabhutosh Kumar · adjusted-price analysis · daily log returns

In [ ]:
# Parameters
ticker = "AAPL"
start_date = "2024-01-01"
end_date = "2024-12-31"
db_path = "data/data.db"

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analysis import (load_stock, calculate_returns, calculate_volatility,
                      calculate_max_drawdown, calculate_sharpe_ratio, metrics, plot_report)
frame = load_stock(db_path, ticker, start_date, end_date)
print(f"{ticker} from {start_date} to {end_date}: {len(frame)} trading days")
print(f"Actual observations: {frame.index.min().date()} to {frame.index.max().date()}")
display(frame.head())
display(frame.tail())
if len(frame) < 21:
    print("Insufficient history for 20 returns: rolling volatility is unavailable.")
if len(frame) < 50:
    print("Insufficient history for a 50-day moving average.")
print("Missing sessions are not filled. Returns span consecutive retained observations; gaps can affect daily-risk estimates.")

## Risk measures
All report returns use adjusted close. Rolling volatility uses 20 log returns and sample standard deviation, annualized by √252. Warm-up gaps are preserved. Sharpe uses the course approximation `(mean log return × 252 − 0.02)/(sample std × √252)`; undefined or zero volatility displays N/A. Drawdown can recover; its minimum is maximum drawdown.

Price segments use mean rolling volatility as a retrospective threshold (red above, green at/below, gray unavailable). Comparisons across stocks with different actual histories are descriptive, not like-for-like performance rankings.

In [ ]:
frame["log_return"] = calculate_returns(frame.adj_close)
frame["rolling_volatility"] = calculate_volatility(frame.log_return)
frame["drawdown"] = calculate_max_drawdown(frame.adj_close)
summary = metrics(frame)
display(frame[["adj_close", "log_return", "rolling_volatility", "drawdown"]].tail())
figure = plot_report(frame, ticker, start_date, end_date)
plt.show()
plt.close(figure)

## Summary
Total return is the endpoint price ratio minus one. Annualized return compounds log returns using 252/n. Whole-period annualized volatility differs from the mean rolling volatility used in the portfolio scatter. Best/worst days and up/down averages are log returns. Positive days include zero-return days in the denominator; empty subsets show N/A.

In [ ]:
labels = {"total_return": "Total Return", "annualized_return": "Annualized Return",
          "annualized_volatility": "Annualized Volatility", "sharpe_ratio": "Sharpe Ratio",
          "max_drawdown": "Maximum Drawdown", "best_day": "Best Day", "worst_day": "Worst Day",
          "positive_days": "Positive Days", "average_up_day": "Average Up Day", "average_down_day": "Average Down Day"}
rows = []
for key, label in labels.items():
    value = summary[key]
    formatted = "N/A" if not np.isfinite(value) else (f"{value:.2f}" if key == "sharpe_ratio" else f"{value:+.2%}")
    rows.append({"Metric": label, "Value": formatted})
display(pd.DataFrame(rows).set_index("Metric"))

In [ ]:
# Named machine-readable result collected only after successful execution.
output = {key: (float(value) if np.isfinite(value) else None) for key, value in summary.items()}
output.update(ticker=ticker, actual_start=str(frame.index.min().date()),
              actual_end=str(frame.index.max().date()), observations=len(frame))
print("HW2_SUMMARY=" + json.dumps(output, allow_nan=False))